In [ ]:
# ==== CONFIG ====
DATASET = "sst2"        # "sst2" (GLUE, binaire) ou "ag_news" (4 classes)
DO_QUICK_FINETUNE = False  # True = fine-tuning rapide sur 1 epoch pour les "base" checkpoints
BATCH = 16
EPOCHS = 1
SEED = 42

# ==== IMPORTS ====
import os, torch, numpy as np
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer)
import evaluate
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# ==== DATA ====
if DATASET.lower() == "sst2":
    ds = load_dataset("glue", "sst2")
    text_col, label_col, num_labels = "sentence", "label", 2
    eval_split = "validation"
    finetuned_models = [
        "distilbert-base-uncased-finetuned-sst-2-english",
        "textattack/roberta-base-SST-2"  # autre checkpoint fine-tuned
    ]
    base_models = [
        "distilbert-base-uncased",
        "bert-base-uncased",
        "roberta-base"
    ]
elif DATASET.lower() == "ag_news":
    ds = load_dataset("ag_news")
    text_col, label_col, num_labels = "text", "label", 4
    eval_split = "test"
    finetuned_models = [
        "textattack/bert-base-uncased-ag-news",
        "valhalla/distilbart-mnli-12-1"  # NLI utilisable en zero-shot (optionnel)
    ]
    base_models = [
        "distilbert-base-uncased",
        "bert-base-uncased",
        "roberta-base"
    ]
else:
    raise ValueError("DATASET doit être 'sst2' ou 'ag_news'")



# Option: petit sous-échantillon pour aller vite (désactive pour un vrai test)
def maybe_small(d, n=500):
    return d.select(range(min(n, d.num_rows)))


train_ds = ds["train"]
eval_ds  = ds[eval_split]
# train_ds, eval_ds = maybe_small(train_ds), maybe_small(eval_ds)  # décommente pour un test ultra-rapide

metric_acc = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro"),
    }

def tokenize_function(examples, tokenizer):
    return tokenizer(examples[text_col], truncation=True)

def evaluate_checkpoint(model_ckpt, eval_only=True):
    tokenizer = AutoTokenizer.from_pretrained(model_ckpt, use_fast=True)
    tokenized_train = train_ds.map(lambda x: tokenize_function(x, tokenizer), batched=True)
    tokenized_eval  = eval_ds.map(lambda x: tokenize_function(x, tokenizer),  batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_ckpt, num_labels=num_labels, ignore_mismatched_sizes=True
    )

    args = TrainingArguments(
        output_dir=f"./runs/{DATASET}/{model_ckpt.split('/')[-1]}",
        evaluation_strategy="epoch",
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        num_train_epochs=EPOCHS if not eval_only else 0.0,
        learning_rate=2e-5,
        weight_decay=0.01,
        seed=SEED,
        logging_steps=50,
        save_strategy="no",
        report_to="none",
    )
 
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_train if not eval_only else None,
        eval_dataset=tokenized_eval,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    if not eval_only:
        trainer.train()

    metrics = trainer.evaluate()
    name = model_ckpt
    print(f"\n== {name} on {DATASET} ==")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")
    return metrics

# ==== ÉVALUATION DES MODÈLES ====
results = {}

print("\n>>> Évaluation des modèles déjà fine-tuned")
for ckpt in finetuned_models:
    try:
        results[ckpt] = evaluate_checkpoint(ckpt, eval_only=True)
    except Exception as e:
        print(f"[WARN] {ckpt} erreur: {e}")

if DO_QUICK_FINETUNE:
    print("\n>>> Fine-tuning rapide (1 epoch) des modèles de base")
    for ckpt in base_models:
        try:
            results[ckpt] = evaluate_checkpoint(ckpt, eval_only=False)
        except Exception as e:
            print(f"[WARN] {ckpt} erreur: {e}")

# Tri et affichage synthétique par accuracy
def extract_acc(m): return float(m.get("eval_accuracy", 0.0))
ranking = sorted(results.items(), key=lambda kv: extract_acc(kv[1]), reverse=True)
print("\n=== RÉSUMÉ (trié par accuracy) ===")
for name, mets in ranking:
    print(f"{name:45s}  acc={extract_acc(mets):.4f}  f1={mets.get('eval_f1_macro',0.0):.4f}")


c:\Users\User\AppData\Local\pypoetry\Cache\virtualenvs\ia-create-ia-cate-0NKycUOr-py3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\User\AppData\Local\pypoetry\Cache\virtualenvs\ia-create-ia-cate-0NKycUOr-py3.12\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\datasets--glue. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need t


>>> Évaluation des modèles déjà fine-tuned


c:\Users\User\AppData\Local\pypoetry\Cache\virtualenvs\ia-create-ia-cate-0NKycUOr-py3.12\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 872/872 [00:

[WARN] distilbert-base-uncased-finetuned-sst-2-english erreur: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'


c:\Users\User\AppData\Local\pypoetry\Cache\virtualenvs\ia-create-ia-cate-0NKycUOr-py3.12\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--textattack--roberta-base-SST-2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 872/872 [00:00<00:00, 45293.2

[WARN] textattack/roberta-base-SST-2 erreur: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

=== RÉSUMÉ (trié par accuracy) ===
